In [ ]:
"""
You can run either this notebook locally (if you have all the dependencies and a GPU) or on Google Colab.

Instructions for setting up Colab are as follows:
1. Open a new Python 3 notebook.
2. Import this notebook from GitHub (File -> Upload Notebook -> "GitHub" tab -> copy/paste GitHub URL)
3. Connect to an instance with a GPU (Runtime -> Change runtime type -> select "GPU" for hardware accelerator)
4. Run this cell to set up dependencies.
5. Restart the runtime (Runtime -> Restart Runtime) for any upgraded packages to take effect


NOTE: User is responsible for checking the content of datasets and the applicable licenses and determining if they are suitable for the intended use.
"""
# If you're using Google Colab and not running locally, run this cell to install dependencies

# Install dependencies
!pip install wget
!apt-get update && apt-get install -y sox libsndfile1 ffmpeg
!pip install text-unidecode
!pip install omegaconf

BRANCH='main'
!python -m pip install git+https://github.com/NVIDIA/NeMo.git@{BRANCH}#egg=nemo_toolkit[all]

In [ ]:
# Optional dependencies:
# - nemo_text_processing: required for Inverse Text Normalization (ITN)
# - vllm: required for LLM-based Speech Translation
#
# For a complete Docker-based setup for speech translation with vLLM, see:
# https://github.com/NVIDIA/NeMo/blob/main/scripts/installers/Dockerfile.speech_translation_vllm

!pip install nemo_text_processing
!pip install vllm==0.12.0

# Introduction

This tutorial introduces the **ASR Inference Pipeline API** — a unified, streaming-first API for speech recognition that lives under `nemo.collections.asr.inference`. It provides a consistent interface for running inference with different ASR models (buffered CTC/RNNT/TDT, cache-aware CTC/RNNT) and optional post-processing features.

## Table of Contents

1. [Dataset Preparation](#Dataset-Preparation)
2. [Pipelines](#Pipelines)
   - [Buffered CTC/RNNT/TDT Pipeline](#Buffered-CTC/RNNT/TDT-Pipeline)
   - [Cache-Aware CTC/RNNT Pipeline](#Cache-Aware-CTC/RNNT-Pipeline)
3. [Advanced Features](#Advanced-Features)
   - [Per-Stream Options](#Per-Stream-Options)
   - [EoU Detection](#EoU-Detection)
   - [Word Timestamps and Confidence Scores](#Word-Timestamps-and-Confidence-Scores)
   - [Per-Stream Biasing](#Per-Stream-Biasing)
   - [Inverse Text Normalization](#Inverse-Text-Normalization)
   - [Speech Translation](#Speech-Translation)
4. [Implementation Details](#Implementation-Details)
   - [Frame](#Frame)
   - [Stream](#Stream)
   - [Multiple Streams](#Multiple-Streams)
   - [Continuous Batching](#Continuous-Batching)

# Dataset Preparation

We use **Mini LibriSpeech** (`dev_clean_2`) — a small subset of LibriSpeech that is also used in other NeMo ASR tutorials. The download script creates a NeMo manifest JSON file that lists every audio file path together with its transcript and duration.

In [ ]:
%%bash
DATA_ROOT="datasets/mini-librispeech"
MANIFEST="$DATA_ROOT/dev_clean_2.json"

if [ -f "$MANIFEST" ]; then
    echo "Dataset already exists at '$DATA_ROOT', skipping download."
else
    echo "Downloading Mini LibriSpeech dataset..."
    mkdir -p "$DATA_ROOT"
    python ../../scripts/dataset_processing/get_librispeech_data.py \
        --data_root "$DATA_ROOT/" \
        --data_sets dev_clean_2
    echo "Download complete."
fi

In [ ]:
import json
import os

MANIFEST_PATH = "datasets/mini-librispeech/dev_clean_2.json"
LIMIT = 5

audio_filepaths = []
reference_texts  = []
with open(MANIFEST_PATH) as f:
    for line in f:
        item = json.loads(line)
        audio_filepaths.append(item["audio_filepath"])
        reference_texts.append(item["text"])
        if len(audio_filepaths) == LIMIT:
            break

print(f"Loaded {len(audio_filepaths)} audio files")
for path, ref in zip(audio_filepaths, reference_texts):
    print(f"  {path.split('/')[-1]}  →  {ref}")

In [ ]:
! wget -nc https://dldata-public.s3.us-east-2.amazonaws.com/2086-149220-0033.wav
! wget -nc https://nemo-public.s3.us-east-2.amazonaws.com/an4_diarize_test.wav

In [ ]:
# Demo files
demo_audio_filepath = "2086-149220-0033.wav"
itn_demo_audio_filepath = "an4_diarize_test.wav"
biasing_demo_audio_file = audio_filepaths[2]

missing = [f for f in [demo_audio_filepath, itn_demo_audio_filepath, biasing_demo_audio_file] if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Demo files not found: {missing} — re-run the download cell above.")
print("All demo files exist.")

# Pipelines

#### How it works?
A `Pipeline` is the central inference engine. It wraps the ASR model, feature extraction, buffer/state/cache management, and optional text post-processing (ITN, translation) into a single object. The primary method is `transcribe_step()`, which accepts a batch of `Frame` objects and immediately returns one `TranscribeStepOutput` per frame.

`PipelineBuilder.build_pipeline(cfg)` is a factory that reads an OmegaConf config and instantiates the appropriate pipeline variant.

#### Key parameters
`TranscribeStepOutput` fields returned by `transcribe_step()`:
- `stream_id`: integer identifying which stream this output belongs to
- `final_transcript`: text finalized at the last EoU boundary; empty on most steps, populated only when EoU is detected
- `partial_transcript`: accumulated text since the previous EoU boundary; updated every step and may change as future frames arrive
- `current_step_transcript`: text decoded from the current frame only
- `final_segments`: list of `TextSegment` objects (word or segment metadata) corresponding to `final_transcript`

#### Warnings and Notes
- `final_transcript` is only non-empty at EoU boundaries. On all other steps it is an empty string — use `partial_transcript` to track in-progress output.
- `partial_transcript` is not stable and will be revised as more audio arrives. Only `final_transcript` should be treated as authoritative output.
- A pipeline session must be opened with `open_session()` before calling `transcribe_step()` and closed with `close_session()` afterward. The convenience method `pipeline.run()` handles this automatically.

In [ ]:
# This cell contains helper imports and utilities

from IPython.display import display, HTML, Audio
from collections import defaultdict
from time import sleep

import re
import gc
import torch
import librosa
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from nemo.collections.asr.inference.factory.pipeline_builder import PipelineBuilder, BasePipeline
from nemo.collections.asr.inference.streaming.framing.request_options import ASRRequestOptions
from nemo.collections.asr.inference.utils.enums import ASROutputGranularity
from nemo.collections.asr.inference.utils.text_segment import TextSegment
from nemo.collections.asr.parts.context_biasing.biasing_multi_model import BiasingRequestItemConfig
from nemo.collections.asr.parts.context_biasing.boosting_graph_batched import BoostingTreeModelConfig
from nemo.collections.asr.parts.utils.eval_utils import remove_punctuations
from nemo.collections.asr.metrics.wer import word_error_rate


_PALETTE = ["#4a9eff", "#ff7043", "#26a69a", "#ab47bc", "#66bb6a", "#ffa726", "#ef5350", "#42a5f5"]


def create_pipeline(config_path: str, cfg_overrides: dict = None):
    cfg = OmegaConf.load(config_path)
    if cfg_overrides:
        _MISSING = object()
        for key, value in cfg_overrides.items():
            if OmegaConf.select(cfg, key, default=_MISSING) is _MISSING:
                raise KeyError(f"cfg_overrides key '{key}' does not exist in config '{config_path}'")
            OmegaConf.update(cfg, key, value)
    return PipelineBuilder.build_pipeline(cfg)


def compute_wer(hypotheses: list[str], references: list[str]):
    """WER with punctuation and capitalisation ignored (mirrors pipeline_eval defaults)."""
    hyps = [remove_punctuations(h).lower() for h in hypotheses]
    refs = [remove_punctuations(r).lower() for r in references]
    return word_error_rate(hypotheses=hyps, references=refs)


def do_streaming(
    pipeline: BasePipeline,
    audio_filepaths: list[str],
    reference_texts: list[str] = None,
    options: list[ASRRequestOptions] = None,
    wait: float = 0.0,
):
    sep = pipeline.get_sep()
    request_generator = pipeline.get_request_generator()

    if options is None:
        options = [ASRRequestOptions() for _ in audio_filepaths]
    request_generator.set_audio_filepaths(audio_filepaths, options)

    has_translation = any(opt.enable_nmt for opt in options)
    COLORS = [_PALETTE[i % len(_PALETTE)] for i in range(len(audio_filepaths))]

    accumulated = defaultdict(str)
    cur_partial = defaultdict(str)
    accumulated_translation = defaultdict(str)
    cur_partial_translation = defaultdict(str)
    pipeline_name = type(pipeline).__name__

    def render_html():
        parts = []
        for i in range(len(audio_filepaths)):
            filename = audio_filepaths[i].split("/")[-1]
            color    = COLORS[i]
            final    = accumulated[i]
            partial  = cur_partial[i]

            meta = f'stream {i} &nbsp;·&nbsp; {pipeline_name} &nbsp;·&nbsp; {filename}'
            if has_translation and options[i].target_language:
                meta += f' &nbsp;·&nbsp; {options[i].source_language} → {options[i].target_language}'

            translation_row = ""
            if has_translation:
                final_tr   = accumulated_translation[i]
                partial_tr = f'<span style="color:#aaa;">{cur_partial_translation[i]}</span>' if cur_partial_translation[i] else ""
                translation_row = (
                    f'<div style="margin-top:6px;padding-top:6px;border-top:1px solid #e0e0e0;'
                    f'color:#555;font-style:italic;">{final_tr}{partial_tr}</div>'
                )

            parts.append(
                f'<div style="margin:4px 0;padding:10px 14px;border-left:4px solid {color};'
                f'background:#fafafa;border-radius:0 4px 4px 0;font-family:serif;">'
                f'<div style="font-size:0.75em;color:#888;margin-bottom:4px;font-family:monospace;">'
                f'{meta}</div>'
                f'<div>{final}{partial}</div>'
                f'{translation_row}'
                f'</div>'
            )
        return "".join(parts)

    handle = display(HTML(render_html()), display_id=True)

    pipeline.open_session()
    for requests in request_generator:
        step_outputs = pipeline.transcribe_step(requests)
        for out in step_outputs:
            sid = out.stream_id
            if out.final_transcript:
                text = out.final_transcript if accumulated[sid] else out.final_transcript.lstrip(sep)
                accumulated[sid] += text
            cur_partial[sid] = out.partial_transcript

            if has_translation:
                if out.final_translation:
                    accumulated_translation[sid] += out.final_translation
                cur_partial_translation[sid] = out.partial_translation

        handle.update(HTML(render_html()))
        if wait > 0:
            sleep(wait)
    pipeline.close_session()

    handle.update(HTML(render_html()))

    if reference_texts is not None:
        hypotheses = [accumulated[i] for i in range(len(audio_filepaths))]
        wer = compute_wer(hypotheses, reference_texts)
        print(f"\nWER for {pipeline_name} is: {wer:.2%}")


def log_text_segments(segments: list[TextSegment], title: str = None):
    has_class = any(getattr(seg, "semiotic_class", None) is not None for seg in segments)

    header_cells = ["#", "Start", "End", "Dur", "Conf"]
    if has_class:
        header_cells.append("Class")
    header_cells.append("Text")

    th = "".join(
        f'<th style="padding:4px 10px;text-align:{"left" if h=="Text" else "right"};'
        f'color:#888;font-weight:normal;border-bottom:1px solid #ddd;">{h}</th>'
        for h in header_cells
    )

    rows = []
    for i, seg in enumerate(segments):
        bg = "#fafafa" if i % 2 == 0 else "#fff"
        cells = [
            f'<td style="padding:3px 10px;text-align:right;color:#aaa;">{i}</td>',
            f'<td style="padding:3px 10px;text-align:right;">{seg.start:.2f}</td>',
            f'<td style="padding:3px 10px;text-align:right;">{seg.end:.2f}</td>',
            f'<td style="padding:3px 10px;text-align:right;">{seg.duration:.2f}</td>',
            f'<td style="padding:3px 10px;text-align:right;">{seg.conf:.3f}</td>',
        ]
        if has_class:
            cls = getattr(seg, "semiotic_class", None) or ""
            cells.append(f'<td style="padding:3px 10px;text-align:right;color:#888;">{cls}</td>')
        cells.append(f'<td style="padding:3px 10px;">{seg.text}</td>')
        rows.append(
            f'<tr style="background:{bg};">{"".join(cells)}</tr>'
        )

    count_label = f'{len(segments)} {"word" if hasattr(segments[0], "word") else "segment"}s' if segments else "0 segments"
    title_html = f'<div style="font-weight:bold;margin-bottom:2px;">{title}</div>' if title else ""
    html = (
        f'<div style="font-family:monospace;font-size:0.88em;margin:6px 0;">'
        f'{title_html}'
        f'<div style="color:#888;margin-bottom:4px;">{count_label}</div>'
        f'<table style="border-collapse:collapse;width:auto;">'
        f'<thead><tr>{th}</tr></thead>'
        f'<tbody>{"".join(rows)}</tbody>'
        f'</table></div>'
    )
    display(HTML(html))


def visualize_word_timestamps(audio_filepath, word_segments):
    samples, sr = librosa.load(audio_filepath, sr=None)
    duration = len(samples) / sr
    t = np.linspace(0, duration, len(samples))

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(t, samples, color="#4a9eff", linewidth=0.4, alpha=0.7)

    colors = plt.cm.tab20.colors
    for i, word in enumerate(word_segments):
        color = colors[i % len(colors)]
        ax.axvspan(word.start, word.end, alpha=0.25, color=color)
        mid = (word.start + word.end) / 2
        y_top = ax.get_ylim()[1]
        ax.text(mid, y_top * 0.88, word.text,
                ha="center", va="top", fontsize=7.5,
                color=color, fontweight="bold", rotation=45)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_title("Audio waveform with word-level timestamps")
    ax.margins(x=0)
    plt.tight_layout()
    plt.show()

    display(Audio(audio_filepath))

## Buffered CTC/RNNT/TDT Pipeline

<img src="./images/buffered_pipeline.png" alt="Buffered Streaming Pipeline" style="max-width:100%;margin:12px 0;" />

#### How it works?
The buffered pipeline adapts any standard offline ASR model (CTC, RNNT, TDT) for streaming without modifying the model architecture. At each step, the pipeline assembles a three-part padded buffer and runs a full forward pass through the model:

- **Left context** — audio from previous steps, providing stable past context so the model "remembers" what came before.
- **Middle chunk** — the newly arrived audio; the primary segment whose tokens will be emitted.
- **Right context** (lookahead) — future audio already received, letting the model resolve ambiguities at the right edge of the middle chunk.

After each forward pass, the pipeline scans the output token sequence looking for **N consecutive blank tokens (EoU)**. When found, all tokens from the start of the middle chunk up to that point are emitted as output.

#### Key parameters
- `left_padding_size`: seconds of past audio prepended as left context
- `chunk_size`: duration of the middle chunk in seconds; also equal to the frame size
- `right_padding_size`: seconds of lookahead audio appended as right context
- `batch_size`: number of streams processed in parallel
- `stateful` (RNNT only): if `True`, the RNNT decoder state is carried over between steps for better transcript continuity; if `False`, the decoder resets at each step

#### Warnings and Notes
- Maximum theoretical latency = `chunk_size` + `right_padding_size`. In the worst case, the pipeline must wait for the full chunk and then buffer the lookahead before emitting tokens.
- Each audio frame is re-encoded together with its left and right context — the same audio samples are processed multiple times; this is the key difference from cache-aware streaming, which processes each frame only once.

In [ ]:
batch_size = 8
log_level = 40

# Amount of past audio prepended as left context so the model can condition on what came before the current chunk
left_padding_size = 1.6

# Duration of the middle chunk. Also, equal to frame size
chunk_size = 0.54

# Amount of future (lookahead) audio appended as right context, helping resolve ambiguities at the right edge of the chunk.
right_padding_size = 1.6

# (RNNT only) - When True, the RNNT decoder state is carried over from the previous step, improving transcript continuity across chunks;
#               when False, the decoder state is reset at each step (stateless decoding).
stateful = True

buffered_rnnt_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/buffered_rnnt.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/parakeet-rnnt-1.1b",
        "streaming.left_padding_size": left_padding_size,
        "streaming.chunk_size": chunk_size,
        "streaming.right_padding_size": right_padding_size,
        "streaming.batch_size": batch_size,
        "streaming.stateful": stateful,
        "asr_decoding_type": "rnnt",
        "log_level": log_level,
    },
)

buffered_ctc_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/buffered_ctc.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/parakeet-ctc-1.1b",
        "streaming.left_padding_size": left_padding_size,
        "streaming.chunk_size": chunk_size,
        "streaming.right_padding_size": right_padding_size,
        "streaming.batch_size": batch_size,
        "asr_decoding_type": "ctc",
        "log_level": log_level,
    },
)

In [ ]:
for pipeline in [buffered_rnnt_pipeline, buffered_ctc_pipeline]:
    do_streaming(pipeline, audio_filepaths, reference_texts)

In [ ]:
del buffered_rnnt_pipeline, buffered_ctc_pipeline
gc.collect()
torch.cuda.empty_cache()

## Cache-Aware CTC/RNNT Pipeline

<img src="./images/cache_aware_pipeline.png" alt="Cache-Aware Streaming Pipeline" style="max-width:100%;margin:12px 0;"/>

#### How it works?
Unlike buffered inference — which slides an overlapping window and re-encodes the same audio frames repeatedly — cache-aware streaming processes **each audio frame once**. The FastConformer encoder maintains a **bounded cache of intermediate representations** across all layers (both self-attention KV states and convolutional states). When a new chunk arrives, only that chunk is encoded; past context is read from the cache rather than recomputed. This eliminates redundant computation and enables stable, predictable memory usage regardless of utterance length.

The chunk size and latency are determined by the attention context configuration `[left_context, right_context]`, where chunk size = `right_context + 1` frames (for FastConformer each frame = 80 ms):
- `[70, 0]` → chunk size = 1 (0.08 s)
- `[70, 1]` → chunk size = 2 (0.16 s)
- `[70, 6]` → chunk size = 7 (0.56 s)
- `[70, 13]` → chunk size = 14 (1.12 s)

#### Key parameters
- `batch_size`: number of concurrent streams processed in each forward pass
- `num_slots`: total number of pre-allocated cache slots; must be greater than or equal to `batch_size`
- `att_context_size`: list `[left_context, right_context]` controlling attention span and chunk size

#### Warnings and Notes
- `num_slots` pre-allocates GPU memory for all cache slots at startup. Set it to the maximum number of concurrent streams expected, not just the current `batch_size`.
- Cache-aware streaming requires a FastConformer model trained with streaming context. Standard offline models are not compatible.
- EoU detection may cause punctuation to be dropped or misplaced at segment boundaries for models trained to emit punctuation after silence.

In [ ]:
batch_size = 8
log_level = 40

# Maximum number of concurrent stream slots in the cache manager. Must be ≥ batch_size. Controls how many slots should be pre-allocated.
num_slots = 64

# Number of frames the self-attention layers can attend to on the left (past) and right (lookahead) sides. 
# Smaller right context lowers latency.
att_context_size = [70, 13]

ca_rnnt_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/cache_aware_rnnt.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/nemotron-speech-streaming-en-0.6b",
        "streaming.batch_size": batch_size,
        "streaming.num_slots": num_slots,
        "streaming.att_context_size": att_context_size,
        "log_level": log_level,
        "asr_decoding_type": "rnnt"
    },
)

ca_ctc_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/cache_aware_ctc.yaml",
    cfg_overrides={
        "asr.model_name": "stt_en_fastconformer_hybrid_large_streaming_multi",
        "streaming.batch_size": batch_size,
        "streaming.num_slots": num_slots,
        "streaming.att_context_size": att_context_size,
        "asr_decoding_type": "ctc",
        "log_level": log_level,
    },
)

In [ ]:
for pipeline in [ca_rnnt_pipeline, ca_ctc_pipeline]:
    do_streaming(pipeline, audio_filepaths, reference_texts)

In [ ]:
del ca_rnnt_pipeline, ca_ctc_pipeline
gc.collect()
torch.cuda.empty_cache()

# Advanced Features

The following sections explore the per-request options and post-processing features available in the Pipeline API. Most examples use a single audio file for clarity, but all features work in batch mode as well.

## Per-Stream Options

#### How it works?
`ASRRequestOptions` is an immutable dataclass that lets you configure inference behaviour **per audio stream**. Every field defaults to `None`, meaning "use the pipeline default". Passing an explicit value overrides the pipeline default for that request only — so different streams in the same batch can have entirely different settings.

#### Key parameters
- `enable_itn`: apply Inverse Text Normalization (e.g. `"twenty dollars"` → `"$20"`)
- `stop_history_eou`: silence window in milliseconds that triggers an EoU boundary; set to `-1` to disable EoU detection
- `asr_output_granularity`: `ASROutputGranularity.SEGMENT` (default) or `ASROutputGranularity.WORD` — controls the granularity of `final_segments` timestamps
- `language_code`: language hint for prompt-enabled multilingual models
- `enable_nmt`: enable Neural Machine Translation post-processing
- `source_language`: NMT source language, effective only when `enable_nmt=True`
- `target_language`: NMT target language, effective only when `enable_nmt=True`
- `biasing_cfg`: per-stream context biasing configuration — key phrases and their boosting weight

#### Warnings and Notes
- `enable_itn` and `enable_nmt` are per-request toggles, but the corresponding modules (ITN grammar, NMT model) must be loaded at **pipeline construction time** via the config. Toggling them per-request without loading at startup has no effect.
- `ASRRequestOptions` is immutable — create a new instance for each request rather than modifying an existing one.

In [ ]:
# All fields are None → pipeline defaults apply for every option
default_opts = ASRRequestOptions()
print("Default options:", default_opts)

# Override individual fields per request
custom_opts = ASRRequestOptions(
    enable_itn=True,                                   # Inverse Text Normalization
    stop_history_eou=400,                              # EoU silence window (ms)
    asr_output_granularity=ASROutputGranularity.WORD,  # word-level timestamps
)
print("Custom options: ", custom_opts)

## EoU Detection

#### How it works?
EoU (End-of-Utterance) detection enables the pipeline to automatically segment a continuous audio stream into discrete, finalized utterances — without requiring explicit stream boundaries from the caller. After each forward pass the pipeline inspects the token sequence in the buffer. When a long-enough run of **silent (blank) tokens** is observed at the trailing edge, the pipeline concludes that the speaker has paused and marks an EoU boundary. At that moment:
- `final_transcript` is populated with all text accumulated since the previous EoU boundary.
- `partial_transcript` is reset to an empty string.

#### Key parameters
- `stop_history_eou` (in `ASRRequestOptions`): silence window in milliseconds; a pause of at least this duration triggers an EoU boundary. Set to `-1` to disable EoU detection entirely.

#### Warnings and Notes
- Different pipeline types implement different EoU detection logic — the exact blank-counting mechanism may vary between buffered and cache-aware pipelines.
- `final_transcript` is an empty string on all steps where no EoU boundary is detected. Always check `partial_transcript` for the current in-progress transcription.
- Setting `stop_history_eou` too low may cause premature EoU boundaries mid-sentence; setting it too high increases latency before a segment is finalized.

In [ ]:
generic_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/buffered_rnnt.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/parakeet-rnnt-1.1b",
        "streaming.left_padding_size": 1.6,
        "streaming.chunk_size": 0.54,
        "streaming.right_padding_size": 1.6,
        "streaming.batch_size": 8,
        "streaming.stateful": True,
        "asr_decoding_type": "rnnt",
        "log_level": 40,
    },
)

In [ ]:
def transcribe_single(pipeline, audio_filepath, options=None, verbose=False):
    """
    Stream a single audio file through *pipeline* and print each EoU event
    together with the partial transcript updates along the way.
    """
    request_generator = pipeline.get_request_generator()
    if options is None:
        options = ASRRequestOptions()
    request_generator.set_audio_filepaths([audio_filepath], [options])

    sep = pipeline.get_sep()
    accumulated_final = ""
    segments = []
    text_to_show = ""

    pipeline.open_session()
    for step, requests in enumerate(request_generator):
        step_outputs = pipeline.transcribe_step(requests)
        for out in step_outputs:
            if out.final_transcript:
                # Strip leading separator on the very first segment
                text = out.final_transcript if accumulated_final else out.final_transcript.lstrip(sep)
                accumulated_final += text + "[EoU-🎬]"

                final_segments = out.final_segments
                if len(final_segments) > 0:
                    # Strip leading separator on the very first segment
                    first_segment = final_segments[0]
                    first_segment.text = first_segment.text.lstrip(sep)
                segments.extend(final_segments)
                    
            partial_transcript = out.partial_transcript
            if verbose:
                text_to_show = f"{accumulated_final}{partial_transcript}".strip()
                print(f"Step#{step:<3} -> {text_to_show}")
    pipeline.close_session()

    final_text = accumulated_final.replace("[EoU-🎬]", "")
    if verbose:
        print("-" * 100)
        print(f"Full transcription: {final_text}")
    return final_text, segments

In [ ]:
# Enable EoU per-request: 400 ms silence window (≈ 5 blank tokens) triggers an EoU boundary
options = ASRRequestOptions(stop_history_eou=400)
final_text, segments = transcribe_single(generic_pipeline, demo_audio_filepath, options=options, verbose=True)

## Word Timestamps and Confidence Scores

Each `TranscribeStepOutput` carries a `final_segments` list that is populated at every EoU boundary. By default the list contains one `TextSegment` per finalized utterance (segment-level granularity). Switching to **word-level** granularity returns one `Word` object per recognized word, enabling precise alignment of each token to its position in the audio timeline.

#### How it works?
Granularity is controlled per-request via `ASRRequestOptions`:

```python
from nemo.collections.asr.inference.utils.enums import ASROutputGranularity
from nemo.collections.asr.inference.streaming.framing.request_options import ASRRequestOptions

# Segment-level (default)
ASRRequestOptions(asr_output_granularity=ASROutputGranularity.SEGMENT)

# Word-level
ASRRequestOptions(asr_output_granularity=ASROutputGranularity.WORD)
```

#### Key parameters
- `asr_output_granularity`: `ASROutputGranularity.SEGMENT` returns one `TextSegment` per EoU segment; `ASROutputGranularity.WORD` returns one `Word` per recognized token with individual start/end timestamps

#### Warnings and Notes
- `final_segments` is only populated at EoU boundaries. It is empty on all other steps.
- Word-level confidence scores are currently only supported for CTC-based pipelines. Confidence scores for RNNT models are under development.

In [ ]:
options = ASRRequestOptions(
    stop_history_eou=400, 
    asr_output_granularity=ASROutputGranularity.SEGMENT
)
_, text_segments = transcribe_single(generic_pipeline, demo_audio_filepath, options=options)
log_text_segments(text_segments, title="Segment-level timestamps")

In [ ]:
options = ASRRequestOptions(
    stop_history_eou=400, 
    asr_output_granularity=ASROutputGranularity.WORD
)
_, word_segments = transcribe_single(generic_pipeline, demo_audio_filepath, options=options)

log_text_segments(word_segments, title="Word-level timestamps")
visualize_word_timestamps(demo_audio_filepath, word_segments)

## Per-Stream Biasing

#### How it works?
Context biasing (phrase boosting) steers the decoder toward a list of expected words or phrases. This is useful when domain-specific terms — product names, proper nouns, technical jargon — are unlikely to be recognized correctly out of the box. A compact token-level prefix tree (boosting tree) is built from the supplied key phrases. At each decoding step the tree contributes a log-probability bonus (`alpha`) to tokens that advance a phrase prefix, raising those tokens above competitors. Each stream in a batch can carry a **different** biasing config, so you can boost different phrases per caller within the same inference pass.

#### Key parameters
- `key_phrases_list` (in `BoostingTreeModelConfig`): list of strings to boost during decoding
- `boosting_model_alpha` (in `BiasingRequestItemConfig`): log-probability bonus applied to tokens that match a boosted phrase prefix; higher values produce stronger boosting

#### Warnings and Notes
- Context biasing currently works with greedy decoding. It will be more effective with beam search, which is under development.
- Setting `boosting_model_alpha` too high can cause the decoder to force a boosted phrase even when it is clearly not present in the audio. Tune this value empirically.
- The boosting tree is built at request time — large key phrase lists with long phrases may add overhead at session startup.

In [ ]:
biasing_options = [
    # No biasing
    ASRRequestOptions(stop_history_eou=400),

    # key_phrases_list=["Yoolka"]
    ASRRequestOptions(
        stop_history_eou=400,
        biasing_cfg=BiasingRequestItemConfig(
            boosting_model_cfg=BoostingTreeModelConfig(key_phrases_list=["Yoolka"]),
            boosting_model_alpha=10.0,
        ),
    ),
    # key_phrases_list=["yoolka", "Antoniya"]
    ASRRequestOptions(
        stop_history_eou=400,
        biasing_cfg=BiasingRequestItemConfig(
            boosting_model_cfg=BoostingTreeModelConfig(key_phrases_list=["yoolka", "Antoniya"]),
            boosting_model_alpha=3.0,
        ),
    ),
]

# pipeline.run() is a convenience wrapper around open_session/transcribe_step/close_session.
# It accepts a list of audio file paths and per-stream options, and returns a list of dicts
# with 'text' and 'segments' keys — one entry per input stream.
# Note: biasing currently works with greedy decoding and will be more effective with beam search.
output = generic_pipeline.run([biasing_demo_audio_file]*len(biasing_options), options=biasing_options)


def highlight_phrases(text, phrases, color="#ff7043"):
    """Wrap each occurrence of a phrase (case-insensitive) in a highlighted span."""
    if not phrases:
        return text
    pattern = re.compile(
        r'\b(' + '|'.join(re.escape(p) for p in phrases) + r')\b',
        re.IGNORECASE,
    )
    return pattern.sub(
        lambda m: (
            f'<span style="background:{color}33;color:{color};font-weight:bold;'
            f'border-radius:3px;padding:1px 5px;">{m.group()}</span>'
        ),
        text,
    )


rows = []
for i, opt in enumerate(biasing_options):
    bcfg = opt.biasing_cfg
    phrases = bcfg.boosting_model_cfg.key_phrases_list if bcfg else []
    keywords_str = ", ".join(f'<code>{p}</code>' for p in phrases) if phrases else "—"
    border_color = "#4a9eff" if phrases else "#ccc"
    highlighted = highlight_phrases(output[i]['text'], phrases)

    rows.append(
        f'<div style="margin:6px 0;padding:10px 14px;border-left:4px solid {border_color};'
        f'background:#fafafa;border-radius:0 4px 4px 0;font-family:serif;">'
        f'<div style="font-size:0.75em;color:#888;margin-bottom:5px;font-family:monospace;">'
        f'stream {i} &nbsp;·&nbsp; bias: {keywords_str}</div>'
        f'<div style="line-height:1.7;">{highlighted}</div>'
        f'</div>'
    )

display(HTML("".join(rows)))

In [ ]:
del generic_pipeline
gc.collect()
torch.cuda.empty_cache()

## Inverse Text Normalization

#### How it works?
**Inverse Text Normalization (ITN)** converts ASR output from spoken form to written form, turning numeric expressions, dates, currency amounts, and other entities into their canonical notation. ITN runs as a post-processing step inside `transcribe_step()` and is applied to each finalized EoU segment before it appears in `final_transcript`. Word-level alignment is preserved, so timestamps remain correct even after normalization.

#### Key parameters
- `enable_itn` (pipeline config): must be `true` at pipeline construction time to compile and load the ITN grammar
- `lang` (pipeline config): language code for the ITN grammar (e.g. `"en"`)

#### Warnings and Notes
- ITN requires the `nemo_text_processing` package with `pynini`, included in the NeMo ASR extras.
- The first run compiles and caches `.far` grammar files to `cache_dir`. Subsequent runs reuse the cache and are fast.
- `enable_itn: true` must be set in the **pipeline config** at startup. Per-request toggling via `ASRRequestOptions` only takes effect when the grammar is already loaded.

In [ ]:
# Build a pipeline with ITN loaded (enable_itn=True in the config).
# The grammar is compiled once at startup; per-request toggling is cheap after that.
itn_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/buffered_rnnt.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/parakeet-rnnt-1.1b",
        "streaming.left_padding_size": 1.6,
        "streaming.chunk_size": 0.54,
        "streaming.right_padding_size": 1.6,
        "streaming.batch_size": 8,
        "streaming.stateful": True,
        "asr_decoding_type": "rnnt",
        "log_level": 40,
        "enable_itn": True,  # compile & load ITN grammar at startup
        "lang": "en",        # language code for the ITN grammar; required when enable_itn=True
    },
)

In [ ]:
# Run the same audio twice in a single batch:
#   stream 0 → ITN OFF   (raw ASR output, spoken form preserved)
#   stream 1 → ITN ON    (spoken numbers/dates/currency converted to written notation)
# For audio that contains numbers, dates, or currency you will see the difference.
itn_options = [
    ASRRequestOptions(enable_itn=False, stop_history_eou=400, asr_output_granularity=ASROutputGranularity.WORD),
    ASRRequestOptions(enable_itn=True,  stop_history_eou=400, asr_output_granularity=ASROutputGranularity.WORD),
]

output = itn_pipeline.run([itn_demo_audio_filepath, itn_demo_audio_filepath], options=itn_options)
for i in range(len(itn_options)):
    itn_enabled = ["OFF", "ON"][int(itn_options[i].enable_itn)]
    log_text_segments(output[i]["segments"], title=f"ITN is {itn_enabled}")

In [ ]:
del itn_pipeline
gc.collect()
torch.cuda.empty_cache()

## Speech Translation

#### How it works?
The Pipeline API supports **LLM-based streaming translation** as an optional post-processing step. It first performs streaming ASR, then simultaneously translates the transcribed source language into any target language. Translation runs automatically inside `transcribe_step()` — no extra calls are needed. Each stream in a batch can target a **different language** independently.

Users will see **partial translations** that may be revised as new audio chunks arrive. After some point, a portion of the translation prefix becomes fixed and will no longer change. The LLM prompt used at each step:
```
Translate the following {src_lang} source text to {tgt_lang}. Always output text in the {tgt_lang} language:
{src_lang}: {asr_predicted_text}
{tgt_lang}: {translation_prefix}
```
The model completes the prompt from `{translation_prefix}`, producing an updated translation consistent with prior output. At each step:

1. Run the LLM on the current ASR transcript, seeded with the translation prefix from the previous step
2. Compare the new translation output with the previous one
3. Extend or retain the prefix using one of two policies:
   - **wait-k**: specifies the maximum number of words the translation is allowed to lag behind the ASR transcript. If the translation falls more than `wait_k` words behind, the prefix is automatically extended so that `len(prefix.split()) >= len(asr_transcript.split()) - wait_k`. Larger values of `wait_k` yield better translation quality at the cost of higher latency — more ASR tokens must arrive before the LLM is invoked.
   - **LCP** (Longest Common Prefix): the prefix is the longest common prefix shared between the current and previous translations. The prefix is never forced to grow. Enable LCP by setting `wait_k=-1`.

#### Key parameters
Per-stream configuration via `ASRRequestOptions`:
- `enable_nmt`: set to `True` to activate translation for this stream
- `source_language`: language of the audio (e.g. `"English"`)
- `target_language`: desired output language (e.g. `"German"`)

Translation output fields in `TranscribeStepOutput`:
- `final_translation`: finalized translation of `final_transcript`, populated at EoU boundaries
- `partial_translation`: running translation of `partial_transcript`, updated every step and may change

#### Warnings and Notes
- `enable_nmt: true` must be set in the **pipeline config** at construction time so the NMT model is loaded. Per-request toggling only works after the model is loaded.
- Translation requires the **vLLM** package: `pip install vllm`
- The NMT model requires significant GPU memory (3+ GB for a 1.7B model). Use `select_devices()` to route ASR and NMT to separate GPUs when available.
- `partial_translation` is not stable and will be revised as more audio arrives. Only `final_translation` should be treated as authoritative translated output.
- Higher `wait_k` values improve BLEU/COMET scores at the cost of increased latency (LAAL).
- `LCP` typically matches or slightly exceeds the best wait-k BLEU score, but also yields the highest latency.
- Better ASR transcripts produce better translations — ASR errors propagate directly into the translation output.

In [ ]:
# del nmt_pipeline if it's already defined to avoid OOM
if "nmt_pipeline" in globals():
    del nmt_pipeline
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
def select_devices(min_single_gpu_gb=80, min_dual_gpu_gb=30):
    """
    Priority:
      1. Single GPU with >= min_single_gpu_gb GB  -> both ASR and NMT on GPU 0
      2. Two GPUs each with >= min_dual_gpu_gb GB -> ASR on GPU 0, NMT on GPU 1
      3. Fallback                                 -> ASR on CPU (device_id=-1), NMT on GPU 0
    """
    n = torch.cuda.device_count()

    def gb(i):
        return torch.cuda.get_device_properties(i).total_memory / 1024 ** 3

    print(f"Available GPUs: {n}")
    for i in range(n):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  —  {gb(i):.1f} GB")

    if n >= 1 and gb(0) >= min_single_gpu_gb:
        asr_device, asr_device_id = "cuda", 0
        nmt_device, nmt_device_id = "cuda", 0
        print(f"\n-> Single large GPU detected. Using GPU 0 ({gb(0):.1f} GB) for both ASR and NMT.")
    elif n >= 2 and gb(0) >= min_dual_gpu_gb and gb(1) >= min_dual_gpu_gb:
        asr_device, asr_device_id = "cuda", 0
        nmt_device, nmt_device_id = "cuda", 1
        print(f"\n-> Two GPUs detected. ASR on GPU 0 ({gb(0):.1f} GB), NMT on GPU 1 ({gb(1):.1f} GB).")
    else:
        asr_device, asr_device_id = "cpu", -1
        nmt_device, nmt_device_id = "cuda", 0
        gpu_info = f"GPU 0 ({gb(0):.1f} GB)" if n >= 1 else "no GPU found"
        print(f"\n-> Insufficient GPU memory. ASR on CPU, NMT on {gpu_info}.")

    return asr_device, asr_device_id, nmt_device, nmt_device_id


asr_device, asr_device_id, nmt_device, nmt_device_id = select_devices()

In [ ]:
nmt_pipeline = create_pipeline(
    "../../examples/asr/conf/asr_streaming_inference/buffered_rnnt.yaml",
    cfg_overrides={
        "asr.model_name": "nvidia/parakeet-rnnt-1.1b",
        "asr.device": asr_device,
        "asr.device_id": asr_device_id,
        "streaming.left_padding_size": 1.6,
        "streaming.chunk_size": 0.54,
        "streaming.right_padding_size": 1.6,
        "streaming.batch_size": 8,
        "streaming.stateful": True,
        "asr_decoding_type": "rnnt",
        "log_level": 40,
        "enable_nmt": True,
        "nmt.model_name": "utter-project/EuroLLM-1.7B-Instruct",
        "nmt.device": nmt_device,
        "nmt.device_id": nmt_device_id,
        "nmt.batch_size": 8,
        "nmt.waitk": -1, 
    },
)

In [ ]:
TARGET_LANGUAGES = ["German", "French", "Spanish", "Russian", "Serbian", "Croatian", "English"]

# Use the same English audio file for each target language
audio_files = [demo_audio_filepath] * len(TARGET_LANGUAGES)

options = [
    ASRRequestOptions(
        enable_nmt=True,
        source_language="English",
        target_language=lang,
        stop_history_eou=400,
    )
    for lang in TARGET_LANGUAGES
]

In [ ]:
do_streaming(nmt_pipeline, audio_files, options=options)

In [ ]:
del nmt_pipeline
gc.collect()
torch.cuda.empty_cache()

# Implementation Details

This section describes the low-level building blocks that underpin the pipeline API. Understanding these concepts is useful for advanced usage, custom integrations, or debugging.

1. **Frame**: An immutable container representing a single chunk of audio.
2. **Stream**: An iterator over an audio source that yields `Frame` objects.
3. **Multiple Streams**: A higher-level wrapper (`MultiStream`) that manages multiple streams simultaneously and interleaves their frames into batches.
4. **Continuous Batching**: Keeps the active batch at full capacity (`ContinuousBatchedFrameStreamer`) by automatically replacing finished streams with new ones.

## Frame

#### How it works?
A `Frame` is a frozen dataclass that wraps a short slice of raw audio samples together with metadata. Frames are the atomic unit of data flowing through the pipeline — each call to `transcribe_step()` receives a batch of `Frame` objects.

#### Key parameters
- `samples`: 1-D Tensor of raw audio samples
- `stream_id`: integer identifier for the audio stream this frame belongs to
- `is_first`: boolean flag marking the first chunk of a stream
- `is_last`: boolean flag marking the last chunk of a stream
- `length`: number of valid samples in the tensor; the remainder may be zero-padding for the last chunk. `-1` means all samples are valid

#### Warnings and Notes
- `Frame` is immutable (frozen dataclass) — do not attempt to modify its fields in-place.

In [ ]:
import torch
from nemo.collections.asr.inference.streaming.framing.request import Frame

# Simulate a 160-ms chunk at 16 kHz
SAMPLE_RATE = 16_000
FRAME_SIZE_SECS = 0.16
n_samples = int(SAMPLE_RATE * FRAME_SIZE_SECS)  # 2560 samples

frame = Frame(
    samples=torch.rand(n_samples),
    stream_id=0,
    is_first=True,
    is_last=False,
    length=n_samples,      # all samples are valid (no padding)
)

print(f"stream_id  : {frame.stream_id}")
print(f"is_first   : {frame.is_first}")
print(f"is_last    : {frame.is_last}")
print(f"length     : {frame.length}")

## Stream

#### How it works?
A `Stream` is an iterator that reads an audio source and emits `Frame` objects. The concrete implementation used here is `MonoStream`, which reads a mono WAV file and slices it into fixed-size, non-overlapping chunks. Each iteration yields a one-element list containing a single `Frame`.

#### Key parameters
- `rate`: target sample rate in Hz; the audio file is resampled automatically if its native rate differs
- `frame_size_in_secs`: duration of each emitted frame in seconds
- `stream_id`: unique integer identifier assigned to every frame produced by this stream
- `pad_last_frame`: if `True`, the final (potentially shorter) frame is zero-padded to the full `frame_size`; the `valid_size` field on that frame reflects the true number of meaningful samples

#### Warnings and Notes
- Frames produced by `MonoStream` do not overlap — each audio sample appears in exactly one frame.
- All frames have the same `size`, but only the last frame may have `valid_size < size`.

In [ ]:
from nemo.collections.asr.inference.streaming.framing.mono_stream import MonoStream

FRAME_SIZE_SECS = 1.0

stream = MonoStream(
    rate=SAMPLE_RATE,
    frame_size_in_secs=FRAME_SIZE_SECS,
    stream_id=0,
    pad_last_frame=True,
)
stream.load_audio(audio_filepaths[0])

# Iterate through all frames to observe size, valid_size, and boundary flags
print(f"Audio file : {audio_filepaths[0]}")
print(f"Frame size : {FRAME_SIZE_SECS}s ({int(SAMPLE_RATE * FRAME_SIZE_SECS)} samples)")
print()

total_frames = 0
for frames_list in stream:
    frame = frames_list[0]   # MonoStream yields a one-element list
    total_frames += 1
    tag = "FIRST" if frame.is_first else ("LAST" if frame.is_last else "")
    print(f"  Frame {total_frames:1d}  size={frame.size:<5d}  valid_size={frame.valid_size:<5d}  {tag}")
print(f"\nTotal frames: {total_frames}")

## Multiple Streams

#### How it works?
When transcribing more than one audio source simultaneously, use `MultiStream` — a higher-level wrapper that manages a fixed collection of `MonoStream` objects and interleaves their frames. At each iteration step it pulls `n_frames_per_stream` frames from every active stream and returns them as a flat batch. Iteration continues until all streams are exhausted; as shorter files finish first, the batch size naturally shrinks.

#### Key parameters
- `n_frames_per_stream`: number of frames drawn from each stream per iteration step

#### Warnings and Notes
- All streams must be added before iteration begins — `MultiStream` does not support adding streams mid-iteration.
- The batch size decreases as shorter streams finish. If you need a consistently full batch throughout, use `ContinuousBatchedFrameStreamer` instead (see the next section).

In [ ]:
from nemo.collections.asr.inference.streaming.framing.multi_stream import MultiStream

multi_streamer = MultiStream(n_frames_per_stream=1)
for audio_id, filepath in enumerate(audio_filepaths):
    stream = MonoStream(
        rate=SAMPLE_RATE,
        frame_size_in_secs=FRAME_SIZE_SECS,
        stream_id=audio_id,
        pad_last_frame=True,
    )
    stream.load_audio(filepath)
    multi_streamer.add_stream(stream=stream, stream_id=stream.stream_id)

print(f"Active streams: {len(multi_streamer)}")
print(f"{'Step':<6} {'Batch':>5}  Stream_ids")
print("-" * 40)

frame_counts = {}
for step, batch in enumerate(multi_streamer):
    # batch is a list of Frame objects.
    # shorter files finish earlier, so the batch size shrinks as streams exhaust.
    for frame in batch:
        frame_counts[frame.stream_id] = frame_counts.get(frame.stream_id, 0) + 1

    ids = [f.stream_id for f in batch]
    print(f"{step:<6} {len(batch):>5}  {ids}")

print(f"\nPer-stream frame counts:")
for sid in sorted(frame_counts):
    print(f"  stream {sid}: {frame_counts[sid]:>3} frames")

## Continuous Batching

#### How it works?
With `MultiStream`, all streams are loaded upfront and the batch shrinks as shorter files finish, leaving GPU capacity underutilised. `ContinuousBatchedFrameStreamer` fixes this by automatically starting a new stream whenever one completes, keeping the active batch at `batch_size` for as long as there are files remaining. It wraps `MultiStream` internally and refills it to capacity at every step.

#### Key parameters
- `sample_rate`: target sample rate in Hz for all streams
- `frame_size_in_secs`: duration of each frame in seconds
- `batch_size`: maximum number of concurrently active streams; new streams are added automatically to maintain this limit
- `n_frames_per_stream`: number of frames pulled from each active stream per iteration step
- `pad_last_frame`: if `True`, the final frame of each stream is zero-padded to the full frame size

#### Warnings and Notes
- The batch size only drops below `batch_size` when fewer than `batch_size` files remain in the queue overall.
- `set_audio_filepaths()` must be called before iteration begins; it accepts a list of file paths and a corresponding list of `ASRRequestOptions`.

In [ ]:
from nemo.collections.asr.inference.streaming.framing.multi_stream import ContinuousBatchedFrameStreamer
from nemo.collections.asr.inference.streaming.framing.request_options import ASRRequestOptions

SAMPLE_RATE = 16_000
FRAME_SIZE_SECS = 1.0
BATCH_SIZE = 3   # at most 3 streams active simultaneously

streamer = ContinuousBatchedFrameStreamer(
    sample_rate=SAMPLE_RATE,
    frame_size_in_secs=FRAME_SIZE_SECS,
    batch_size=BATCH_SIZE,
    n_frames_per_stream=1,
    pad_last_frame=True,
)

# define per-stream options
options = [ASRRequestOptions() for _ in audio_filepaths]
streamer.set_audio_filepaths(audio_filepaths, options)

print(f"Total files  : {len(audio_filepaths)}")
print(f"Batch size   : {BATCH_SIZE}  (active streams)")
print()
print(f"{'Step':<6} {'Batch':>5}  Stream_ids")
print("-" * 40)

frame_counts = {}
for step, batch in enumerate(streamer):
    # batch shrinks only when fewer than batch_size files remain overall;
    # otherwise it stays at batch_size even as individual streams finish.
    for frame in batch:
        frame_counts[frame.stream_id] = frame_counts.get(frame.stream_id, 0) + 1

    ids = [f.stream_id for f in batch]
    print(f"{step:<6} {len(batch):>5}  {ids}")

print(f"\nPer-stream frame counts:")
for sid in sorted(frame_counts):
    print(f"  stream {sid}: {frame_counts[sid]:>3} frames")